# Validación de modelos

Hemos construido un modelo. Pero, ¿qué tan bueno es?

En esta lección, aprenderemos a usar la validación del modelo para medir la calidad de nuestro modelo. Medir la calidad del modelo es la clave para mejorar nuestros modelos de manera iterativa.

## ¿Qué es la validación de modelos?

Siempre vamos a querer evaluar casi todos los modelos que construyamos. En la mayoría de las aplicaciones (aunque no en todas), la medida relevante de la calidad del modelo es la precisión predictiva. En otras palabras, ¿las predicciones del modelo estarán cerca de lo que realmente sucede?

Muchas personas cometen un gran error al medir la precisión predictiva. Hacen predicciones con sus datos de entrenamiento y comparan esas predicciones con los valores objetivo en los datos de entrenamiento. Veremos el problema con este enfoque y cómo solucionarlo en un momento, pero primero pensemos en cómo lo haríamos.

Primero necesitaríamos resumir la calidad del modelo de una manera comprensible. Si comparamos los valores predichos y reales de 10,000 casas, probablemente encontraremos una mezcla de buenas y malas predicciones. Revisar una lista de 10,000 valores predichos y reales sería inútil. Necesitamos resumir esto en una sola métrica.

Existen muchas métricas para resumir la calidad del modelo, pero comenzaremos con una llamada Error Absoluto Medio (Mean Absolute Error, también conocido como MAE). Desglosaremos esta métrica comenzando con la última palabra, error.

El error de predicción para cada casa es:

$$
error = real − predicción
$$

Entonces, si una casa costó 150,000 y predijiste que costaría 100,000, el error es 50,000.

Con la métrica MAE, tomamos el valor absoluto de cada error. Esto convierte cada error en un número positivo. Luego tomamos el promedio de esos errores absolutos. Esta es nuestra medida de la calidad del modelo. En lenguaje sencillo, se puede decir:

> En promedio, nuestras predicciones están equivocadas por aproximadamente X.

Para calcular el MAE, primero necesitamos un modelo.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

# Cargar datos
ruta = './data/melb_data.csv'
melbourne_data = pd.read_csv(ruta) 

# Quitar nulos
melbourne_data_filtrada = melbourne_data.dropna(axis=0)

# Elegir target
y = melbourne_data_filtrada.Price

# Elegir features
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']

# Crear X
X = melbourne_data_filtrada[melbourne_features]


# Definir modelo
modelo = DecisionTreeRegressor()

# Ajustar modelo
modelo.fit(X, y)

,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_node

Ya tenemos nuestro modelo y ahora podemos calcular el MAE. Para esto utilizaremos también `sklearn`.

In [2]:
from sklearn.metrics import mean_absolute_error

predicciones = modelo.predict(X)
mean_absolute_error(y, predicciones)

434.71594577146544

## El Problema con los puntajes "dentro de la muestra"

La medida que acabamos de calcular puede llamarse un puntaje "dentro de la muestra". Usamos una sola "muestra" de casas tanto para construir el modelo como para evaluarlo. Esto está mal. Veamos por qué.

Imaginemos que en el mercado inmobiliario el color de la puerta no está relacionado con el precio de la casa.

Sin embargo, en la muestra de datos que usamos para construir el modelo, todas las casas con puertas verdes eran muy caras. El trabajo del modelo es encontrar patrones que predigan los precios de las casas, así que verá este patrón y siempre predecirá precios altos para las casas con puertas verdes.

Dado que este patrón se derivó de los datos de entrenamiento, el modelo parecerá preciso en los datos de entrenamiento.

Pero si este patrón no se mantiene cuando el modelo ve nuevos datos, el modelo sería muy impreciso cuando se use en la práctica.

Dado que el valor práctico de los modelos proviene de hacer predicciones sobre nuevos datos, medimos el rendimiento en datos que no se usaron para construir el modelo. La forma más sencilla de hacer esto es excluir algunos datos del proceso de construcción del modelo y luego usarlos para probar la precisión del modelo en datos que no ha visto antes. Estos datos se llaman datos de validación.

## Partiendo los datos
Vamos a dividir nuestros datos en 2 subconjuntos: set de entrenamiento y set de pruebas.

`scikit-learn` tiene una función llamada `train_test_split` para dividir los datos en estas dos partes. Usaremos algunos de esos datos como datos de entrenamiento para ajustar el modelo, y usaremos los otros datos como datos de validación para calcular el error absoluto medio (`mean_absolute_error`).

In [3]:
from sklearn.model_selection import train_test_split

Dividimos los datos en datos de entrenamiento y de validación, tanto para las características como para el objetivo

Esta división se basa en un generador de números aleatorios. Por lo tanto, podemos darle un `random_state` para que siempre que corramos nuestro código, tengamamos exactamente los mismos sets de entrenamiento y de pruebas.

La función `train_test_split` recibe 2 argumentos y 1 opcional. El primer argumento es el dataframe que contiene los features, y el segundo argumento contiene los precios (targets). 

La función nos regresará 4 conjuntos de datos. 2 para entrenar y 2 para probar

In [4]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)

In [5]:
train_X.head()

,Rooms,Bathroom,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude
10385,3,1.0,206.0,110.0,1980.0,-37.87107,145.04991
5805,2,1.0,0.0,73.0,2000.0,-37.85900,144.97670
8488,2,1.0,2701.0,79.0,2011.0,-37.81090,144.86840
6672,3,1.0,670.0,116.0,1940.0,-37.81340,144.87450
776,6,3.0,708.0,275.0,1988.0,-37.91810,145.04400


In [6]:
train_y.head()

10385    1060000.0
5805      390000.0
8488      502000.0
6672     1055000.0
776      1900000.0
Name: Price, dtype: float64

In [7]:
val_X.head()

,Rooms,Bathroom,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude
4850,2,1.0,96.0,71.0,1880.0,-37.85010,144.99530
2307,2,1.0,0.0,70.0,1965.0,-37.89020,144.99070
10090,2,1.0,136.0,58.0,1892.0,-37.85542,144.99571
3645,3,2.0,205.0,184.0,1995.0,-37.79930,145.02670
4930,2,1.0,400.0,88.0,1955.0,-37.73520,144.98520


In [8]:
val_y.head()

4850      815000.0
2307      655000.0
10090     957500.0
3645     1330000.0
4930      722000.0
Name: Price, dtype: float64

Ahora definamos nuestro modelo nuevamento y ajustémoslo

In [9]:
modelo = DecisionTreeRegressor()

# Ajustar únicamente con datos de entrenamiento
modelo.fit(train_X, train_y)

# obtener predicciones
val_predictions = modelo.predict(val_X)
print(mean_absolute_error(val_y, val_predictions))

258520.04260813427


**¡¿Ves la diferencia?!**

Cuando validamos nuestro modelo con los mismos datos con los que lo entrenamos, obtubimos un error de alrededor 500 dólares, pero ahora tenemos un error de 263,515 dólares!

---

Ésta es la diferencia entre un modelo que es casi exactamente correcto y uno que es inutilizable para la mayoría de los propósitos prácticos. Como punto de referencia, el valor promedio de una casa en los datos de validación es de 1.1 millones de dólares. Por lo tanto, el error en los nuevos datos es aproximadamente una cuarta parte del valor promedio de la casa.

Hay muchas maneras de mejorar este modelo, como experimentar para encontrar mejores características o diferentes tipos de modelos.